# 03 -- SQLAlchemy avance

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- configurer un moteur et une session SQLAlchemy 2.x ;
- declarer des modeles avec `DeclarativeBase` et `Mapped` ;
- gerer les relations one-to-many, many-to-many ;
- interroger avec `select()` et les filtres ;
- executer des transactions et rollbacks ;
- utiliser les `relationship`, le lazy loading et les join ;
- faire des agregations et group by ;
- migrer avec Alembic (intro) ;
- utiliser l'API Core pour des requetes brutes.

## Prerequis -- ce que vous devez maitriser

- SQL de base (SELECT, INSERT, JOIN) ;
- les type hints, classes Python, fonctions typees ;
- le context manager `with`.

Ce que ce notebook **ne couvrira pas** :

- Flask-SQLAlchemy (couche d'adaptation) ;
- l'async avec `asyncpg` ;
- SQLAlchemy 1.x (API legacy).

## Plan

1. Installation et architecture
2. Engine et session
3. Modele declaratif
4. Creer des tables
5. Inserer et committer
6. Requetes select
7. Filtres et operateurs
8. Update et delete
9. Relations one-to-many
10. Relations many-to-many
11. Joins et eager loading
12. Agregations
13. Transactions
14. API Core et SQL brut
15. Alembic : migrations
16. Synthese
17. Exercices

---

## 1. Installation et architecture

In [ ]:
# pip install sqlalchemy
import sqlalchemy as sa
sa.__version__

### Architecture SQLAlchemy 2.x

- **Engine** : pool de connexions vers la DB
- **Session** : unite de travail (transactions, identity map)
- **ORM** : mapping classes <-> tables via `DeclarativeBase`
- **Core** : abstraction bas niveau, requetes expressives

---

## 2. Engine et session

In [ ]:
from sqlalchemy import create_engine
engine = create_engine('sqlite:///:memory:', echo=False)
engine

In [ ]:
from sqlalchemy.orm import Session
with Session(engine) as session:
    print(type(session).__name__)

`echo=True` affiche toutes les requetes SQL generees (utile pour le debug).

---

## 3. Modele declaratif

In [ ]:
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from sqlalchemy import String

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = 'users'
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    email: Mapped[str] = mapped_column(String(100), unique=True)

User.__table__

---

## 4. Creer des tables

In [ ]:
Base.metadata.create_all(engine)
print('Tables crees :', Base.metadata.tables.keys())

---

## 5. Inserer et committer

In [ ]:
with Session(engine) as s:
    alice = User(name='Alice', email='alice@example.com')
    s.add(alice)
    s.commit()
    print('id =', alice.id)

In [ ]:
with Session(engine) as s:
    s.add_all([
        User(name='Bob', email='bob@example.com'),
        User(name='Eve', email='eve@example.com'),
    ])
    s.commit()
    print('OK')

---

## 6. Requetes select

In [ ]:
from sqlalchemy import select

with Session(engine) as s:
    users = s.scalars(select(User)).all()
    for u in users:
        print(u.id, u.name, u.email)

In [ ]:
# Un seul
with Session(engine) as s:
    u = s.scalars(select(User).where(User.name == 'Alice')).first()
    print(u.email if u else None)

---

## 7. Filtres et operateurs

In [ ]:
with Session(engine) as s:
    stmt = select(User).where(User.name.like('%e%'))
    for u in s.scalars(stmt):
        print(u.name)

In [ ]:
from sqlalchemy import or_, and_

with Session(engine) as s:
    stmt = select(User).where(or_(User.name == 'Alice', User.name == 'Bob'))
    print([u.name for u in s.scalars(stmt)])

In [ ]:
# in_
with Session(engine) as s:
    stmt = select(User).where(User.name.in_(['Alice', 'Eve']))
    print([u.name for u in s.scalars(stmt)])

In [ ]:
# Order by et limit
with Session(engine) as s:
    stmt = select(User).order_by(User.name).limit(2)
    print([u.name for u in s.scalars(stmt)])

---

## 8. Update et delete

In [ ]:
with Session(engine) as s:
    u = s.scalars(select(User).where(User.name == 'Bob')).one()
    u.email = 'bob@new.com'
    s.commit()
    print('Mis a jour')

In [ ]:
# Update en bulk
from sqlalchemy import update
with Session(engine) as s:
    s.execute(update(User).where(User.name == 'Eve').values(email='eve@new.com'))
    s.commit()
    print('OK')

In [ ]:
from sqlalchemy import delete
with Session(engine) as s:
    s.execute(delete(User).where(User.name == 'Bob'))
    s.commit()
    print('Bob supprime')

---

## 9. Relations one-to-many

In [ ]:
from sqlalchemy import ForeignKey
from sqlalchemy.orm import relationship

class Base2(DeclarativeBase):
    pass

class Author(Base2):
    __tablename__ = 'authors'
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    books: Mapped[list['Book']] = relationship(back_populates='author')

class Book(Base2):
    __tablename__ = 'books'
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(100))
    author_id: Mapped[int] = mapped_column(ForeignKey('authors.id'))
    author: Mapped['Author'] = relationship(back_populates='books')

engine2 = create_engine('sqlite:///:memory:')
Base2.metadata.create_all(engine2)
print('OK')

In [ ]:
with Session(engine2) as s:
    tolkien = Author(name='Tolkien')
    tolkien.books = [Book(title='LOTR'), Book(title='The Hobbit')]
    s.add(tolkien)
    s.commit()

with Session(engine2) as s:
    a = s.scalars(select(Author)).first()
    print(a.name, [b.title for b in a.books])

---

## 10. Relations many-to-many

In [ ]:
from sqlalchemy import Table, Column, Integer

class Base3(DeclarativeBase):
    pass

student_course = Table(
    'student_course', Base3.metadata,
    Column('student_id', Integer, ForeignKey('students.id')),
    Column('course_id', Integer, ForeignKey('courses.id')),
)

class Student(Base3):
    __tablename__ = 'students'
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    courses: Mapped[list['Course']] = relationship(secondary=student_course, back_populates='students')

class Course(Base3):
    __tablename__ = 'courses'
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(100))
    students: Mapped[list['Student']] = relationship(secondary=student_course, back_populates='courses')

engine3 = create_engine('sqlite:///:memory:')
Base3.metadata.create_all(engine3)
print('OK')

In [ ]:
with Session(engine3) as s:
    py = Course(title='Python')
    js = Course(title='JS')
    alice = Student(name='Alice', courses=[py, js])
    bob = Student(name='Bob', courses=[py])
    s.add_all([alice, bob])
    s.commit()

with Session(engine3) as s:
    for st in s.scalars(select(Student)):
        print(st.name, '->', [c.title for c in st.courses])

---

## 11. Joins et eager loading

In [ ]:
from sqlalchemy.orm import joinedload, selectinload

with Session(engine2) as s:
    stmt = select(Author).options(selectinload(Author.books))
    for a in s.scalars(stmt):
        print(a.name, len(a.books))

In [ ]:
# Join explicite
with Session(engine2) as s:
    stmt = select(Book.title, Author.name).join(Author)
    for row in s.execute(stmt):
        print(row)

---

## 12. Agregations

In [ ]:
from sqlalchemy import func
with Session(engine2) as s:
    stmt = select(Author.name, func.count(Book.id)).join(Book).group_by(Author.id)
    for row in s.execute(stmt):
        print(row)

---

## 13. Transactions

In [ ]:
try:
    with Session(engine) as s:
        s.add(User(name='Carol', email='carol@x.com'))
        s.commit()
        # Si exception -> rollback automatique
except Exception as e:
    print('Rollback :', e)

with Session(engine) as s:
    print('Carol :', s.scalars(select(User).where(User.name == 'Carol')).first())

In [ ]:
# Savepoint
with Session(engine) as s:
    with s.begin_nested():
        s.add(User(name='Dave', email='dave@x.com'))
        # s.rollback()  # rollback du savepoint seulement
    s.commit()
    print('Dave added')

---

## 14. API Core et SQL brut

In [ ]:
from sqlalchemy import text
with engine.connect() as conn:
    rows = conn.execute(text('SELECT name FROM users ORDER BY name'))
    for r in rows:
        print(r)

In [ ]:
# Parametres lies (protection SQL injection)
with engine.connect() as conn:
    r = conn.execute(text('SELECT * FROM users WHERE name = :n'), {'n': 'Alice'})
    print(list(r))

---

## 15. Alembic : migrations

**Alembic** est l'outil officiel de migrations SQLAlchemy.

```bash
pip install alembic
alembic init migrations
alembic revision --autogenerate -m 'add user table'
alembic upgrade head
```

Le fichier `env.py` doit importer vos modeles pour que `--autogenerate` detecte les changements.

---

## Synthese

| Operation | SQLAlchemy 2.x |
|---|---|
| Engine | `create_engine('sqlite:///:memory:')` |
| Session | `with Session(engine) as s:` |
| Modele | `class X(Base): id: Mapped[int] = mapped_column(primary_key=True)` |
| Insert | `s.add(obj)`, `s.commit()` |
| Select | `s.scalars(select(X).where(...))` |
| Update | `s.execute(update(X).where(...).values(...))` |
| Delete | `s.execute(delete(X).where(...))` |
| Relation 1-N | `relationship(back_populates=...)` + `ForeignKey` |
| Eager | `.options(selectinload(X.rel))` |
| Aggregation | `func.count`, `.group_by` |

### Regles a retenir

- **SQLAlchemy 2.x = syntaxe `select(...).where(...)`** (pas `query(...)`).
- **Un commit par unite de travail** logique.
- **`with Session(engine)` = rollback auto** si exception.
- **Toujours bind les parametres** (`:param`), jamais de f-string SQL.
- **Alembic** pour les migrations en production.

---

## Exercices

### Exercice 1 (facile) -- Modele Product

Definissez une classe `Product` avec `id`, `name`, `price` et creez la table en memoire.

In [ ]:
from sqlalchemy import create_engine, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

# TODO

<details>
<summary>Voir la correction</summary>

```python
from sqlalchemy import create_engine, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
    pass

class Product(Base):
    __tablename__ = 'products'
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    price: Mapped[float]

engine = create_engine('sqlite:///:memory:')
Base.metadata.create_all(engine)
print('OK')
```

</details>

### Exercice 2 (facile) -- CRUD simple

Avec la classe `Product` precedente, inserez 3 produits, listez-les par prix decroissant.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from sqlalchemy.orm import Session
from sqlalchemy import select

with Session(engine) as s:
    s.add_all([
        Product(name='Livre', price=15.0),
        Product(name='Pomme', price=1.0),
        Product(name='Pull', price=40.0),
    ])
    s.commit()

with Session(engine) as s:
    for p in s.scalars(select(Product).order_by(Product.price.desc())):
        print(p.name, p.price)
```

</details>

### Exercice 3 (moyen) -- Relation Category -> Product

Definissez une relation 1-N : un `Category` a plusieurs `Product`. Creez 2 categories et 4 produits. Affichez pour chaque categorie ses produits.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from sqlalchemy import create_engine, String, ForeignKey, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

class Base(DeclarativeBase):
    pass

class Category(Base):
    __tablename__ = 'categories'
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    products: Mapped[list['Product']] = relationship(back_populates='category')

class Product(Base):
    __tablename__ = 'products'
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    category_id: Mapped[int] = mapped_column(ForeignKey('categories.id'))
    category: Mapped['Category'] = relationship(back_populates='products')

engine = create_engine('sqlite:///:memory:')
Base.metadata.create_all(engine)

with Session(engine) as s:
    livres = Category(name='Livres', products=[Product(name='Python'), Product(name='Rust')])
    meubles = Category(name='Meubles', products=[Product(name='Table'), Product(name='Chaise')])
    s.add_all([livres, meubles])
    s.commit()

with Session(engine) as s:
    for cat in s.scalars(select(Category)):
        print(cat.name, '->', [p.name for p in cat.products])
```

</details>

### Exercice 4 (moyen) -- Agregation

Comptez le nombre de produits par categorie avec `func.count` + `group_by`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from sqlalchemy import func, select
from sqlalchemy.orm import Session

with Session(engine) as s:
    stmt = select(Category.name, func.count(Product.id)).join(Product).group_by(Category.id)
    for row in s.execute(stmt):
        print(row)
```

</details>

### Exercice 5 (difficile) -- Transaction avec rollback

Implementez une fonction `transfer(from_id, to_id, amount)` qui retire un montant d'un compte et l'ajoute a un autre, avec rollback si le solde devient negatif.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

class Base(DeclarativeBase):
    pass

class Account(Base):
    __tablename__ = 'accounts'
    id: Mapped[int] = mapped_column(primary_key=True)
    balance: Mapped[float]

engine = create_engine('sqlite:///:memory:')
Base.metadata.create_all(engine)

with Session(engine) as s:
    s.add_all([Account(id=1, balance=100.0), Account(id=2, balance=50.0)])
    s.commit()

def transfer(from_id: int, to_id: int, amount: float) -> bool:
    try:
        with Session(engine) as s:
            src = s.get(Account, from_id)
            dst = s.get(Account, to_id)
            src.balance -= amount
            dst.balance += amount
            if src.balance < 0:
                raise ValueError('solde insuffisant')
            s.commit()
            return True
    except ValueError:
        return False

print('Transfer 30 :', transfer(1, 2, 30))
print('Transfer 200:', transfer(1, 2, 200))
```

</details>

---

## Ressources externes

- [Documentation SQLAlchemy 2.x](https://docs.sqlalchemy.org/en/20/)
- [Unified Tutorial](https://docs.sqlalchemy.org/en/20/tutorial/index.html)
- [Alembic](https://alembic.sqlalchemy.org/)